# ETF Tools — User Guide

Two modules for exploring and analysing UCITS ETF data scraped from justETF.

```
etf_selector.py          — search the full ETF universe by country / sector / holding / theme
portfolio_exposure.py    — given a list of ISINs, compute cumulative country, sector & holding exposure
```

```
etf_selector.py
│
├── Data
│   └── load_data(path)                    → pd.DataFrame
│
├── Discovery
│   ├── find_by_country(country, df, ...)  → pd.DataFrame   rank ETFs by country allocation
│   ├── find_by_sector(sector, df, ...)    → pd.DataFrame   rank ETFs by sector allocation
│   ├── find_by_holding(company, df, ...)  → pd.DataFrame   ETFs containing a stock
│   ├── find_by_theme(keyword, df, ...)    → pd.DataFrame   keyword search on name/index/category
│   ├── find_by_combo(df, ...)             → pd.DataFrame   multi-filter AND search
│   └── find_by_holdings_multi(...)        → dict[str, df]  batch holding search
│
└── Utilities
    ├── list_countries(df)                 → list[str]
    └── list_sectors()                     → list[str]

portfolio_exposure.py
│
└── analyze(isins, weights, budget, ...)   → None   prints country / sector / holding tables
```

| Function | Needs API key | Best for |
|---|---|---|
| `find_by_country` | no | "which ETFs give me Japan?" |
| `find_by_sector` | no | "which ETFs are heavy on Tech?" |
| `find_by_holding` | no | "which ETFs hold NVIDIA?" |
| `find_by_theme` | no | "find AI / robotics / dividend ETFs" |
| `find_by_combo` | no | AND-filter across country + sector + theme |
| `find_by_holdings_multi` | no | screen a list of companies in one call |
| `analyze` | no | portfolio overlap / exposure breakdown |

## Sections
1. [Setup](#1-setup)
2. [Load data](#2-load-data)
3. [find_by_country](#3-find_by_country)
4. [find_by_sector](#4-find_by_sector)
5. [find_by_holding](#5-find_by_holding)
6. [find_by_theme](#6-find_by_theme)
7. [find_by_combo](#7-find_by_combo)
8. [find_by_holdings_multi & NON_US_COMPARABLES](#8-find_by_holdings_multi)
9. [Portfolio exposure — analyze()](#9-portfolio-exposure)
10. [End-to-end: build & analyse a portfolio](#10-end-to-end)


## 1 — Setup

In [ ]:
import sys, os

# Make sure the project root is on the path when running from a subdirectory
sys.path.insert(0, os.path.abspath("."))

import pandas as pd

from etf_selector import (
    load_data,
    find_by_country,
    find_by_sector,
    find_by_holding,
    find_by_theme,
    find_by_combo,
    find_by_holdings_multi,
    list_countries,
    list_sectors,
    NON_US_COMPARABLES,
)
from portfolio_exposure import analyze, load_index

pd.set_option("display.max_colwidth", 55)
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.width", 160)

print("Imports OK")


## 2 — Load data

Both modules read from the scraped data in `data/`:

| File | Format | Used by |
|---|---|---|
| `data/profiles.jsonl` | newline-delimited JSON | `etf_selector.load_data()` |
| `data/profiles.json` | JSON array | `portfolio_exposure.load_index()` |

`load_data()` returns a flat `pd.DataFrame` — one row per ETF — with nested list columns (`country_exposure`, `sector_exposure`, `top_holdings`) that the search functions iterate over internally.


In [ ]:
df = load_data()          # default path: data/profiles.jsonl
print(f"ETFs loaded : {len(df):,}")
print(f"Columns     : {list(df.columns)}")
print(f"Countries   : {len(list_countries(df))} unique")
print(f"Sectors     : {list_sectors()}")


## 3 — find_by_country

Ranks ETFs by their allocation to a single country.

**Parameters**

| Param | Default | Description |
|---|---|---|
| `country` | required | Country name exactly as stored (use `list_countries(df)` to browse) |
| `min_weight` | `5.0` | Minimum country allocation % — ETFs below this are excluded |
| `top_n` | `15` | Max rows returned |
| `print_results` | `True` | Print a formatted table; set `False` to get the DataFrame silently |

Returns a `pd.DataFrame` sorted by `country_weight_pct` desc, then `quality_score` desc.  
`quality_score` = bonus for large AUM − penalty for high TER.


In [ ]:
# ETFs with at least 10% allocation to Japan
result = find_by_country("Japan", df, min_weight=10, top_n=5)


In [ ]:
# Silent mode — get the DataFrame without printing
result_silent = find_by_country("Germany", df, min_weight=5, top_n=10, print_results=False)
print(f"Found {len(result_silent)} ETFs with >=5% Germany")
print(result_silent[["isin", "name", "country_weight_pct"]].head(5).to_string(index=False))


## 4 — find_by_sector

Ranks ETFs by their allocation to a GICS sector.

**Valid sectors** (pass exactly as shown):  
`Basic Materials`, `Consumer Discretionary`, `Consumer Staples`, `Energy`, `Financials`,  
`Health Care`, `Industrials`, `Real Estate`, `Technology`, `Telecommunication`, `Utilities`

**Parameters**

| Param | Default | Description |
|---|---|---|
| `sector` | required | Sector name (case-insensitive match) |
| `min_weight` | `10.0` | Minimum sector allocation % |
| `top_n` | `15` | Max rows returned |
| `print_results` | `True` | Print formatted table |


In [ ]:
# ETFs with >30% in Technology
find_by_sector("Technology", df, min_weight=30, top_n=5)


In [ ]:
# Unknown sector — the function prints valid options and returns empty DataFrame
find_by_sector("Defence", df)


## 5 — find_by_holding

Finds ETFs that contain a specific stock in their **top holdings** list.  
Matching is a case-insensitive substring search — partial names work fine.

**Parameters**

| Param | Default | Description |
|---|---|---|
| `company` | required | Company name or fragment, e.g. `"NVIDIA"`, `"Samsung"`, `"lam "` |
| `top_n` | `15` | Max ETFs returned, sorted by `holding_weight_pct` desc |
| `print_results` | `True` | Print formatted table |

> **Note:** only top-holdings are scraped (typically 5–15 positions per ETF). A company not appearing here may still be held at a lower weight.


In [ ]:
# Which ETFs hold NVIDIA, and how much?
find_by_holding("NVIDIA", df, top_n=8)


In [ ]:
# Partial match — finds "Lam Research", "Lam Research Corp", etc.
find_by_holding("lam ", df, top_n=5)


## 6 — find_by_theme

Keyword search across ETF `name`, `index`, `investment_focus`, and `category` fields.  
Best for thematic / narrative searches when you don't know specific holdings.

**Parameters**

| Param | Default | Description |
|---|---|---|
| `keyword` | required | Any substring, e.g. `"AI"`, `"semiconductor"`, `"dividend"`, `"clean energy"` |
| `top_n` | `15` | Max results, sorted by `fund_size_eur_mln` desc |
| `print_results` | `True` | Print formatted table |

Unlike the other functions, results are sorted by **AUM** (largest fund first) since there is no per-ETF weight to rank by.


In [ ]:
find_by_theme("semiconductor", df, top_n=6)


In [ ]:
find_by_theme("defence", df, top_n=6)


## 7 — find_by_combo

AND-filter: returns only ETFs that satisfy **all** non-`None` criteria simultaneously.  
Internally calls `find_by_country`, `find_by_sector`, `find_by_theme` silently and intersects the ISIN sets.

**Parameters**

| Param | Default | Description |
|---|---|---|
| `country` | `None` | Country filter (same rules as `find_by_country`) |
| `sector` | `None` | Sector filter (same rules as `find_by_sector`) |
| `theme` | `None` | Theme keyword (same rules as `find_by_theme`) |
| `min_country_weight` | `5.0` | Passed to the country sub-query |
| `min_sector_weight` | `10.0` | Passed to the sector sub-query |
| `top_n` | `15` | Max results |

Results are sorted by AUM desc. At least one filter must be non-`None`.


In [ ]:
# ETFs with South Korea exposure AND Technology sector
find_by_combo(df, country="South Korea", sector="Technology", min_sector_weight=10)


In [ ]:
# ETFs that are thematically "AI" AND have US exposure
find_by_combo(df, country="United States", theme="AI", min_country_weight=30, top_n=6)


## 8 — find_by_holdings_multi & NON_US_COMPARABLES

`find_by_holdings_multi` runs `find_by_holding` over a list of companies in a single call — useful when screening a set of stocks for ETF coverage.

**Parameters**

| Param | Default | Description |
|---|---|---|
| `companies` | required | `list[str]` or `list[(name, country_label)]` — country label is display-only |
| `top_n` | `8` | Max ETFs per company |
| `print_results` | `True` | Print per-company sections |

**Returns** `dict[company_name → pd.DataFrame]` — each DataFrame has columns `isin`, `name`, `matched_holding`, `holding_weight_pct`.

---

`NON_US_COMPARABLES` is a pre-built list of 25 non-US companies comparable to the holdings found in typical AI/tech/infrastructure ETF portfolios, grouped by theme:

- **Semiconductors**: Infineon, STMicro, NXP, Tokyo Electron, Advantest, MediaTek, ...
- **Defence / Space**: Rheinmetall, Leonardo, BAE Systems, Saab, Hensoldt, ...
- **Networking**: Ericsson
- **Energy / Grid**: Siemens Energy, Siemens, Vestas, RWE, ...
- **Industrial**: Atlas Copco, CNH Industrial, ...


In [ ]:
# Screen a custom set of European defence companies
defence = [
    ("Rheinmetall", "DE"),
    ("Thales", "FR"),
    ("Leonardo", "IT"),
    ("BAE Systems", "UK"),
    ("Saab", "SE"),
    ("Hensoldt", "DE"),
]
results = find_by_holdings_multi(defence, df, top_n=3)


In [ ]:
# Use the return value — e.g. inspect ETFs holding Rheinmetall
rheinmetall_etfs = results["Rheinmetall"]
print(rheinmetall_etfs[["isin", "name", "holding_weight_pct"]].to_string(index=False))


In [ ]:
# Run the full pre-built NON_US_COMPARABLES list (top 2 ETFs per company)
find_by_holdings_multi(NON_US_COMPARABLES, df, top_n=2)


## 9 — Portfolio exposure — analyze()

`analyze()` takes a list of ISINs (your selected portfolio), optionally custom weights, and a total budget, then prints three weighted-aggregate tables:

- **Country exposure** — e.g. "61% US, 4% Japan"
- **Sector exposure** — e.g. "38% Technology, 22% Industrials"
- **Top holdings** — individual stocks ranked by weighted contribution

**Parameters**

| Param | Default | Description |
|---|---|---|
| `isins` | required | List of ISINs; duplicates are silently dropped |
| `weights` | `None` | List of floats (normalised automatically). `None` = equal weight |
| `budget` | `10_000.0` | Total investment in EUR — used to compute EUR amounts |
| `top_n` | `15` | Rows per table |
| `data_path` | `"data/profiles.json"` | Path to the profiles JSON array |

**Output** is printed to stdout (no return value).  
Uses `data/profiles.json` (the array format), not the JSONL used by `etf_selector`.


In [ ]:
# Equal-weight portfolio — 10 ETFs, 10,000 EUR
MY_ISINS = [
    "IE000XJA2OU4",   # Franklin ClearBridge US Smaller Companies
    "IE000FF2EBQ8",   # BNP Paribas ECPI Global ESG Infrastructure
    "IE000G0E83X3",   # iShares AI Innovation Active
    "IE000LGWDNE5",   # Invesco AI Enablers
    "IE00BKTLJC87",   # iShares Smart City Infrastructure
    "IE0000ZL1RD2",   # Global X AI Semiconductor & Quantum
    "IE00BP3QZD73",   # iShares MSCI World Mid-Cap Equal Weight
    "IE000AON7ET1",   # ARK Space & Defence Innovation
    "IE000Y9MG996",   # Amundi US Tech 100 Equal Weight
    "IE000NX8S1Z1",   # Lunate Boreas S&P AI Data & Infrastructure
]

analyze(MY_ISINS, budget=10_000)


In [ ]:
# Custom weights — overweight the two infrastructure ETFs
analyze(
    MY_ISINS,
    weights=[1, 2, 1, 1, 2, 1, 1, 1, 1, 1],  # relative weights, auto-normalised
    budget=10_000,
    top_n=10,
)


## 10 — End-to-end: discover → screen → analyse

A realistic workflow: find ETFs with European defence exposure, check which non-US companies they hold, then run portfolio exposure on the shortlist.


In [ ]:
def build_portfolio(df, country=None, sector=None, theme=None, top_n=5):
    """
    Step 1 — discover candidate ETFs via find_by_combo.
    Returns a list of ISINs.
    """
    candidates = find_by_combo(
        df,
        country=country,
        sector=sector,
        theme=theme,
        top_n=top_n,
        print_results=False,
    )
    if candidates.empty:
        print("No candidates found.")
        return []
    print(f"Candidates ({len(candidates)}):")
    for _, r in candidates.iterrows():
        print(f"  {r['isin']}  {r['name'][:60]}")
    return list(candidates["isin"])


# Step 1 — find ETFs themed "defence" with Germany exposure
portfolio_isins = build_portfolio(df, country="Germany", theme="defence", top_n=6)


In [ ]:
# Step 2 — screen the shortlist for non-US comparable companies
if portfolio_isins:
    shortlist_df = df[df["isin"].isin(portfolio_isins)]
    find_by_holdings_multi(
        [("Rheinmetall", "DE"), ("Hensoldt", "DE"), ("Saab", "SE"), ("Leonardo", "IT")],
        shortlist_df,
        top_n=5,
    )


In [ ]:
# Step 3 — combine with your existing AI/tech ISINs and analyse the full portfolio
combined = MY_ISINS + portfolio_isins   # duplicates dropped inside analyze()

if combined:
    analyze(combined, budget=15_000, top_n=12)
